# Financial Performance & Profitability Analysis
### Exploratory Data Analysis & Diagnostic Financial Modeling
**Author:** Senior Data Analyst & BI Specialist  
**Tech Stack:** Python (pandas, numpy, matplotlib, seaborn), SQL, Power BI, DAX  
**Dataset:** Cleaned Microsoft Financial Dataset & Strategic Budget Model  
---

## 1. Executive Context & Objectives
This notebook simulates an in-depth corporate financial diagnosis for executive management. The goal is to evaluate corporate revenue growth, understand the direct cost structure (COGS & commercial discounts), identify pockets of value erosion, assess regional and product-line margins, and analyze budget variance targets.

### Core Business Questions Addressed:
1. What is the overall corporate revenue, cost base, and realized gross profit?
2. What are the monthly revenue and gross margin trajectories over the 16-month period?
3. Which customer segments and product lines drive sustainable margins versus value destruction?
4. What is the impact of promotional discount bands on gross profit erosion?
5. How do regional operating territories compare in revenue vs margin conversion?
6. Are commercial business units meeting their budget targets?


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Libraries imported successfully.')


## 2. Ingesting Cleaned Financial Datasets
We load the validated, clean financial dataset and the illustrative budget model from ../data/processed/.

In [ ]:
data_path = '../data/processed/financial_sales_cleaned.csv'
budget_path = '../data/processed/simulated_budget.csv'

df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
df_budget = pd.read_csv(budget_path)

print(f'Financial Sales Transactions: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Budget Target Records:        {df_budget.shape[0]} rows, {df_budget.shape[1]} columns')
df.head()


## 3. High-Level Corporate Financial KPIs
We calculate total Gross Sales, Discounts Granted, Net Revenue, Cost of Goods Sold (COGS), and Gross Profit.

In [ ]:
kpis = {
    'Total Gross Sales ($)': df['gross_sales'].sum(),
    'Total Discounts ($)': df['discounts'].sum(),
    'Total Net Revenue ($)': df['net_sales'].sum(),
    'Total COGS ($)': df['cogs'].sum(),
    'Total Gross Profit ($)': df['gross_profit'].sum(),
    'Overall Gross Margin (%)': (df['gross_profit'].sum() / df['net_sales'].sum()) * 100,
    'Overall Discount Rate (%)': (df['discounts'].sum() / df['gross_sales'].sum()) * 100
}

pd.DataFrame(list(kpis.items()), columns=['Financial Metric', 'Value'])


## 4. Monthly Financial Trajectory & Margin Stability
How did revenue, COGS, and profit margins evolve across the 16 operating months (Sep 2013 - Dec 2014)?

In [ ]:
monthly = df.groupby(['year', 'month_number', 'month_name']).agg(
    units_sold=('units_sold', 'sum'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

monthly['date_sort'] = pd.to_datetime(monthly['year'].astype(str) + '-' + monthly['month_number'].astype(str) + '-01')
monthly = monthly.sort_values(by='date_sort').reset_index(drop=True)
monthly['period_label'] = monthly['date_sort'].dt.strftime('%b %Y')
monthly['margin_pct'] = (monthly['gross_profit'] / monthly['net_sales']) * 100
monthly['mom_rev_growth_%'] = monthly['net_sales'].pct_change() * 100

monthly[['period_label', 'net_sales', 'cogs', 'gross_profit', 'margin_pct', 'mom_rev_growth_%']]


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

x = np.arange(len(monthly))
w = 0.35
ax1.bar(x - w/2, monthly['net_sales'] / 1e6, w, label='Net Revenue ()', color='#3B82F6', alpha=0.85)
ax1.bar(x + w/2, monthly['cogs'] / 1e6, w, label='COGS ()', color='#64748B', alpha=0.85)
ax1.set_ylabel('Revenue & Costs ($ Millions)', fontsize=12, fontweight='bold', color='#1E293B')
ax1.set_xticks(x)
ax1.set_xticklabels(monthly['period_label'], rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(x, monthly['margin_pct'], color='#10B981', marker='o', linewidth=2.5, label='Gross Margin %')
ax2.set_ylabel('Gross Margin (%)', fontsize=12, fontweight='bold', color='#10B981')
ax2.set_ylim(0, 25)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Monthly Revenue, COGS, and Margin Trajectory (Sep 2013 - Dec 2014)', fontsize=14, fontweight='bold', pad=15)
plt.show()


## 5. Commercial Segment Diagnosis: The Enterprise Loss Anomaly
A major finding from our initial audit is that certain segments generate negative gross profit. Let us break down commercial performance by customer segment.

In [ ]:
seg_perf = df.groupby('segment').agg(
    transactions=('transaction_id', 'count'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

seg_perf['gross_margin_%'] = (seg_perf['gross_profit'] / seg_perf['net_sales']) * 100
seg_perf['revenue_share_%'] = (seg_perf['net_sales'] / seg_perf['net_sales'].sum()) * 100
seg_perf['profit_share_%'] = (seg_perf['gross_profit'] / seg_perf['gross_profit'].sum()) * 100

seg_perf.sort_values(by='gross_profit', ascending=False)


### Root Cause Analysis: Enterprise Discount Leakage
Why is Enterprise operating at a -,545 loss despite .61M in revenue?

In [ ]:
ent_drill = df[df['segment'] == 'Enterprise'].groupby('discount_band').agg(
    tx_count=('transaction_id', 'count'),
    units=('units_sold', 'sum'),
    gross_sales=('gross_sales', 'sum'),
    discounts=('discounts', 'sum'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

ent_drill['realized_margin_%'] = (ent_drill['gross_profit'] / ent_drill['net_sales']) * 100
ent_drill['avg_price_per_unit'] = ent_drill['net_sales'] / ent_drill['units']
ent_drill['avg_cost_per_unit'] = ent_drill['cogs'] / ent_drill['units']

ent_drill


## 6. Product Economics & Margin Performance
Evaluating the six product lines across revenue, COGS, units sold, and margins.

In [ ]:
prod_perf = df.groupby('product').agg(
    units_sold=('units_sold', 'sum'),
    mfg_price=('manufacturing_price', 'first'),
    gross_sales=('gross_sales', 'sum'),
    discounts=('discounts', 'sum'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

prod_perf['gross_margin_%'] = (prod_perf['gross_profit'] / prod_perf['net_sales']) * 100
prod_perf['avg_realized_price'] = prod_perf['net_sales'] / prod_perf['units_sold']
prod_perf['avg_cost_per_unit'] = prod_perf['cogs'] / prod_perf['units_sold']

prod_perf.sort_values(by='gross_profit', ascending=False)


## 7. Geographic Regional Profitability
Evaluating geographic revenue and margin performance across Canada, France, Germany, Mexico, and USA.

In [ ]:
cntry_perf = df.groupby('country').agg(
    units_sold=('units_sold', 'sum'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

cntry_perf['gross_margin_%'] = (cntry_perf['gross_profit'] / cntry_perf['net_sales']) * 100
cntry_perf['profit_share_%'] = (cntry_perf['gross_profit'] / cntry_perf['gross_profit'].sum()) * 100
cntry_perf.sort_values(by='gross_profit', ascending=False)


## 8. Budget vs Actual Variance Analysis
Comparing actual performance against management budget targets by business unit.

In [ ]:
dim_seg = pd.read_csv('../data/processed/dim_segment.csv')
b_merged = df_budget.merge(dim_seg, on='segment_id')

b_summary = b_merged.groupby('segment_name').agg(
    budget_revenue=('budget_revenue', 'sum'),
    budget_profit=('budget_gross_profit', 'sum')
).reset_index()

a_summary = df.groupby('segment').agg(
    actual_revenue=('net_sales', 'sum'),
    actual_profit=('gross_profit', 'sum')
).reset_index().rename(columns={'segment': 'segment_name'})

bva = b_summary.merge(a_summary, on='segment_name')
bva['rev_variance ($)'] = bva['actual_revenue'] - bva['budget_revenue']
bva['rev_variance (%)'] = (bva['rev_variance ($)'] / bva['budget_revenue']) * 100
bva['target_achievement (%)'] = (bva['actual_revenue'] / bva['budget_revenue']) * 100
bva['profit_variance ($)'] = bva['actual_profit'] - bva['budget_profit']

bva.sort_values(by='rev_variance ($)', ascending=False)


## 9. Key Analytical Findings & Business Conclusions
1. **Corporate Margin Baseline:** Total revenue of **.73M** generated **.89M in Gross Profit** (14.23% gross margin).
2. **Enterprise Value Destruction:** The Enterprise segment generated **-.55K in gross losses** due to severe contractual discounting. High and Medium discounts push realized sale price below unit COGS.
3. **Government Anchor:** Government sales account for **44.2% of corporate revenue** and **67.4% of total profit** (.39M, 21.69% margin).
4. **Channel Partner Efficiency:** While only generating .80M in revenue (1.5%), Channel Partners achieved the highest margin at **73.13%**, representing a high-margin expansion opportunity.
5. **US Geographic Lag:** Despite generating the highest revenue (.03M), the United States delivered the lowest margin (11.97%), driven by heavy Enterprise discounting.
